In [1]:
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely.ops import transform
import cartopy.feature as cfeature
import matplotlib.patheffects as pe
import textwrap
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import xarray as xr
import glob

In [2]:
# read in basin definition file
polygons_dict = {}

# read in basin definition file
with open("tc_basins.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if basin_name not in polygons_dict:
            polygons_dict[basin_name] = []
        polygons_dict[basin_name].append(poly)

# Convert to GeoDataFrame
basin_records = []

for name, poly_list in polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    basin_records.append({
        "basin name": name,
        "geometry": geom
    })

basins = gpd.GeoDataFrame(basin_records, crs="EPSG:4326")

# fix invalid polygons
basins["geometry"] = basins["geometry"].buffer(0)

# remove empy geometries
basins = basins[~basins.geometry.is_empty]

# convert basins' lon to -180-180
basins["geometry"] = basins["geometry"].apply(
    lambda geom: transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )
)

# read in NAtl subbasin polygons
sub_polygons_dict = {}

with open("tc_subbasins_NAtl_v5.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        sub_basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lon_vals = [(lon + 180) % 360 - 180 for lon in lon_vals]
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if sub_basin_name not in sub_polygons_dict:
            sub_polygons_dict[sub_basin_name] = []
        sub_polygons_dict[sub_basin_name].append(poly)

# Convert to GeoDataFrame
sub_basin_records = []

for name, poly_list in sub_polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    sub_basin_records.append({
        "sub_basin_name": name,
        "geometry": geom
    })

sub_basins = gpd.GeoDataFrame(sub_basin_records, crs="EPSG:4326",geometry="geometry")

# fix invalid polygons
sub_basins["geometry"] = sub_basins["geometry"].buffer(0)

# remove empty geometries
sub_basins = sub_basins[~sub_basins.geometry.is_empty]

# longitude conversion
import shapely.ops
def shift_lon(geom):
    return shapely.ops.transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )

# shift lon
sub_basins["geometry"] = sub_basins["geometry"].apply(shift_lon)

In [7]:
# now calc specific humidity

# load air temp dataset
air = xr.open_mfdataset("datasets/potential_intensity/input/air_temp_mean_1979-2025_daily.nc")

# load rhum dataset
rhum = xr.open_dataset("datasets/potential_intensity/input/rhum_mean_1979-2025_daily.nc")

print(rhum)

<xarray.Dataset> Size: 1GB
Dimensions:      (time: 7161, p: 17, lat: 37, lon: 73)
Coordinates:
  * time         (time) datetime64[ns] 57kB 1979-06-01 1979-06-02 ... 2025-10-01
  * p            (p) float32 68B 1e+03 925.0 850.0 700.0 ... 50.0 30.0 20.0 10.0
  * lat          (lat) float32 148B 90.0 87.5 85.0 82.5 80.0 ... 7.5 5.0 2.5 0.0
  * lon          (lon) float32 292B -160.0 -157.5 -155.0 ... 15.0 17.5 20.0
    spatial_ref  int64 8B ...
Data variables:
    rhum         (time, p, lat, lon) float32 1GB ...


In [9]:
# temperature in Kelvin
T = air["air"]

# convert Kelvin to Celsius
T = T - 273.15

# relative humidity as fraction (0-1)
RH = rhum["rhum"] / 100

# saturation vapor pressure (hPa)
es = 6.112 * np.exp(
    (17.67 * T) / (T + 243.5)
)

# actual vapor pressure (hPa)
e = RH * es

# pressure coordinate (hPa)
p = air.p

# mixing ratio (kg/kg)
r = 0.622 * e / (p - e)

# convert to g/kg for tcpyPI
R = r * 1000

print("\nMixing ratio:")
print("min:", R.min(skipna=True).values)
print("max:", R.max(skipna=True).values)
print("mean:", R.mean(skipna=True).values)
print("NaNs:", R.isnull().sum().values)

# # save to dataset
# R.to_netcdf("datasets/potential_intensity/input/mixing_ration_mean_1979-2025_daily.nc")

print(R)


Mixing ratio:
min: 0.0
max: 33.903458
mean: 1.6807194
NaNs: 65981454
<xarray.DataArray (time: 7161, p: 17, lat: 37, lon: 73)> Size: 1GB
dask.array<mul, shape=(7161, 17, 37, 73), dtype=float32, chunksize=(7161, 17, 37, 73), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 57kB 1979-06-01 1979-06-02 ... 2025-10-01
  * p            (p) float32 68B 1e+03 925.0 850.0 700.0 ... 50.0 30.0 20.0 10.0
  * lat          (lat) float32 148B 90.0 87.5 85.0 82.5 80.0 ... 7.5 5.0 2.5 0.0
  * lon          (lon) float32 292B -160.0 -157.5 -155.0 ... 15.0 17.5 20.0
    spatial_ref  int64 8B 0
Attributes: (12/14)
    precision:           2
    dataset:             NCEP/DOE AMIP-II Reanalysis (Reanalysis-2) Daily Ave...
    level_desc:          Pressure Levels
    statistic:           Mean
    parent_stat:         Individual Obs
    cell_methods:        time: mean (of 4 6-hourly values in one day)
    ...                  ...
    long_name:           Level
    positive:         